# La curva que no miente: 65 años respirando CO₂ en Mauna Loa
## Serie temporal + cambio estructural — ¿Cuándo y cuánto se aceleró el planeta?

**Pregunta central:** ¿Hay evidencia estadística de una aceleración en la tasa de crecimiento del CO₂ atmosférico post-2000, y cuál es la magnitud de ese cambio estructural?

---

## 1. Contexto: el registro que no debía existir (½ cuartilla)

1958. Charles Keeling, un geoquímico de 30 años, instala un analizador infrarrojo en la cima del Mauna Loa (3.400 m, Hawái). Su jefe piensa que es una pérdida de tiempo: "el CO₂ está bien mezclado, mídelo en cualquier lado". Keeling insiste. La financiación casi se cancela dos veces.

Hoy, esa terquedad nos dio **el registro continuo más largo de la atmósfera terrestre**. La Curva de Keeling no es solo datos: es el latido del sistema climático. El dataset utilizado en este proyecto proviene del monitoreo mensual de NOAA/ESRL en Mauna Loa (`co2_mm_mlo.csv`), con columnas para fecha decimal, concentración promedio (`co2`), serie interpolada (`interpolated`), tendencia desestacionalizada (`trend`) y conteo de días de medición. Esa estructura permite separar tres capas del fenómeno: variabilidad estacional (respiración de la biosfera), tendencia de largo plazo (acumulación antropogénica) y ruido de medición/ausencias.

La pregunta que nos trae aquí no es "¿sube el CO₂?" —eso ya lo sabemos—. La pregunta es: **¿la *tasa* de crecimiento se ha acelerado?** Una pendiente constante implicaría emisiones lineales; una aceleración (pendiente creciente) sugeriría retroalimentaciones del ciclo de carbono (permafrost, incendios, saturación de sumideros) o simple insuficiencia de políticas de mitigación. Por eso el análisis combina descomposición STL, detección de quiebres estructurales y modelos de tendencia/forecast: queremos pasar de la intuición visual de la curva a una afirmación cuantitativa sobre el régimen de crecimiento.

**Hipótesis formales:**
- H₀: La tasa de crecimiento es constante (pendiente β₁ = 0 en modelo cuadrático, o sin cambio estructural en pendiente lineal por tramos)
- H₁: Existe un cambio estructural (breakpoint) con Δpendiente > 0 post-breakpoint

**Parámetros de interés:** Δpendiente (ppm/año²), fecha del breakpoint (año), magnitud del cambio en ppm/año, forecast 2030 con IC 95%.

## 2. Dataset: proveniencia, estructura, calidad y sesgos (½ cuartilla)

In [ ]:
# Carga y descripción del dataset
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/raw/co2_mm_mlo.csv")
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
print(f"Rango temporal: {df['year'].min():.0f}-{df['year'].max():.0f}" if 'year' in df.columns else "")
print(f"Missing CO2 (col 'co2'): {df['co2'].isna().sum()}")
if 'interpolated' in df.columns:
    print(f"Missing interpolated: {df['interpolated'].isna().sum()}")
df.head(10)


In [ ]:
# Diccionario de variables y calidad
print("\n=== DICCIONARIO DE VARIABLES ===")
dic = {
    'year': 'Año (entero)',
    'month': 'Mes (1-12)',
    'decimal_date': 'Fecha decimal (año + fracción)',
    'co2': 'CO₂ promedio mensual (ppm) - dato observado',
    'interpolated': 'Serie mensual suavizada/interpolada usada para modelado',
    'trend': 'Componente/tendencia desestacionalizada (si disponible)',
    'days': 'Días de medición en el mes (si disponible)'
}
for k, v in dic.items():
    if k in df.columns:
        print(f"  {k:15s}: {v}")

# Construcción robusta del índice temporal
if {'year', 'month'}.issubset(df.columns) and pd.api.types.is_integer_dtype(df['month']) and df['month'].between(1,12).all():
    idx = pd.to_datetime(df[['year','month']].assign(day=1))
elif 'decimal_date' in df.columns:
    dd = pd.to_numeric(df['decimal_date'], errors='coerce')
    years = np.floor(dd).astype(int)
    months = np.clip(((dd - years) * 12 + 1).round().astype(int), 1, 12)
    idx = pd.to_datetime(dict(year=years, month=months, day=1))
    if not {'year','month'}.issubset(df.columns):
        df['year'] = years
        df['month'] = months
else:
    idx = pd.RangeIndex(start=0, stop=len(df), step=1)

value_col = 'interpolated' if 'interpolated' in df.columns else ('trend' if 'trend' in df.columns else 'co2')
series = pd.to_numeric(df[value_col], errors='coerce')
series.index = idx
series = series.sort_index().interpolate(limit_direction='both')
series.name = 'co2_ppm'

if 'month' in df.columns and pd.api.types.is_numeric_dtype(df['month']) and df['month'].between(1,12).all():
    miss_by_month = df.assign(_month=df['month'].astype(int)).groupby('_month')['co2'].apply(lambda x: x.isna().sum())
    print(f"\nMissing CO2 por mes:\n{miss_by_month}")

print(f"\nSerie lista: {len(series)} meses, {series.index[0].date() if hasattr(series.index[0], 'date') else series.index[0]} a {series.index[-1].date() if hasattr(series.index[-1], 'date') else series.index[-1]}")


**Fuente:** NOAA/ESRL Global Monitoring Laboratory, Mauna Loa Observatory.
**Metodología:** Mediciones NDIR (infrarrojo no dispersivo) calibradas vs estándares WMO.
**Missingness:** ~10% de meses (principalmente invierno) por condiciones meteorológicas; NOAA interpola usando splines cúbicos + ajuste armónico.
**Sesgos conocidos:**
- Registro puntual (Mauna Loa, 3.4 km elevación) ≠ promedio global (diferencia ~1-2 ppm)
- Interpolación puede suavizar variabilidad extrema
- Cambios instrumentales (1974, 1983, 2000s) documentados en metadata NOAA

**Decisión de preprocesamiento:** Usamos `interpolated` (columna oficial NOAA sin gaps) para modelado principal; validamos robustez con `co2` (datos brutos) en análisis de sensibilidad.

## 3. Descomponamos la respiración del planeta (¾ cuartilla)

In [ ]:
import sys
from pathlib import Path
_ROOT = Path('..').resolve()
for _p in [str(_ROOT), str(_ROOT / 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 1. Serie temporal cruda + STL decomposition
import sys
sys.path.append("../src")
from stats_helpers import stl_decomposition, growth_rate_annual
import matplotlib.pyplot as plt
from src.viz import setup_plotting_style, save_fig

setup_plotting_style()

stl = stl_decomposition(series, period=12, seasonal=13, robust=True)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(stl.index, stl['observed'], color='#1f77b4', linewidth=0.5)
axes[0].set_ylabel('CO₂ (ppm)')
axes[0].set_title('CO₂ Mauna Loa: Serie observada')
axes[1].plot(stl.index, stl['trend'], color='#d62728', linewidth=1)
axes[1].set_ylabel('Tendencia')
axes[2].plot(stl.index, stl['seasonal'], color='#2ca02c', linewidth=0.5)
axes[2].set_ylabel('Estacionalidad')
axes[3].plot(stl.index, stl['resid'], color='#7f7f7f', linewidth=0.3)
axes[3].set_ylabel('Residuo')
axes[3].set_xlabel('Año')
plt.tight_layout()
save_fig("01_stl_decomposition")
plt.show()

In [ ]:
# 2. Tasa de crecimiento anual (diff 12 meses) + LOESS
growth = growth_rate_annual(series, periods=12).dropna()
growth.name = 'growth_ppm_yr'

from scipy.signal import savgol_filter
# Suavizado LOESS (usamos Savitzky-Golay como proxy rápido)
growth_smooth = pd.Series(savgol_filter(growth.values, window_length=25, polyorder=3),
                          index=growth.index, name='growth_smooth')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(growth.index, growth.values, alpha=0.3, linewidth=0.5, label='Crecimiento anual (bruto)')
ax.plot(growth_smooth.index, growth_smooth.values, color='#d62728', linewidth=2, label='Suavizado (Savitzky-Golay)')
ax.axhline(growth_smooth.loc[:'1999'].mean(), color='#1f77b4', linestyle='--', label=f'Media pre-2000: {growth_smooth.loc[:"1999"].mean():.2f} ppm/año')
ax.axhline(growth_smooth.loc['2000':].mean(), color='#ff7f0e', linestyle='--', label=f'Media post-2000: {growth_smooth.loc["2000":].mean():.2f} ppm/año')
ax.set_ylabel('Crecimiento CO₂ (ppm/año)')
ax.set_title('Tasa de crecimiento interanual del CO₂')
ax.legend()
plt.tight_layout()
save_fig("01_growth_rate")
plt.show()

print(f"Crecimiento medio 1980-1999: {growth_smooth.loc['1980':'1999'].mean():.3f} ppm/año")
print(f"Crecimiento medio 2000-2023: {growth_smooth.loc['2000':].mean():.3f} ppm/año")
print(f"Diferencia: {growth_smooth.loc['2000':].mean() - growth_smooth.loc['1980':'1999'].mean():.3f} ppm/año")

In [ ]:
# 3. Test de breakpoint: PELT (ruptures) / Bai-Perron (statsmodels)
from stats_helpers import changepoint_pelt, changepoint_bai_perron

# PELT en la tendencia (sin estacionalidad)
trend = stl['trend'].dropna()
breaks_pelt = []
try:
    breaks_pelt = changepoint_pelt(trend, penalty=15, model='rbf')
    print(f"PELT breakpoints (índices): {breaks_pelt}")
    if breaks_pelt:
        print(f"PELT breakpoints (fechas): {[trend.index[min(i-1, len(trend)-1)].date() for i in breaks_pelt[:-1]]}")
except Exception as e:
    print(f"PELT no disponible/falló ({e}). Se usará Bai-Perron o breakpoint de referencia.")
    breaks_pelt = []

# Bai-Perron (statsmodels)
bp_breaks = []
try:
    bp = changepoint_bai_perron(trend, max_breaks=3, h=0.1)
    bp_breaks = list(getattr(bp, 'breaks', []) or [])
    print(f"Bai-Perron breaks (índices): {bp_breaks}")
    if bp_breaks:
        print(f"Bai-Perron fechas: {[trend.index[min(i, len(trend)-1)].date() for i in bp_breaks]}")
except Exception as e:
    print(f"Bai-Perron falló: {e}")

# Selección robusta de breakpoint (índice en trend)
bp_idx = None
if breaks_pelt and len(breaks_pelt) >= 2:
    bp_idx = int(breaks_pelt[-2])
elif bp_breaks:
    bp_idx = int(bp_breaks[0])
else:
    # Fallback cercano al 2000 si los métodos no están disponibles
    target = trend.index.get_indexer([pd.Timestamp('2000-01-01')], method='nearest')[0]
    bp_idx = int(target)
    print(f"Usando breakpoint de referencia (fallback): {trend.index[bp_idx].date()}")
bp_idx = max(12, min(bp_idx, len(trend)-13))
print(f"Breakpoint seleccionado (índice): {bp_idx} -> {trend.index[bp_idx].date()}")


In [ ]:
# 4. Cuantificación: pendiente pre/post 2000 con IC bootstrap
from stats_helpers import ic_media_bootstrap, t_test_welch

# Pendiente anual por OLS en ventanas móviles de 10 años
def rolling_slope(s, window=120):  # 10 años = 120 meses
    slopes = []
    for i in range(len(s) - window):
        y = s.iloc[i:i+window].values
        x = np.arange(window)
        slope = np.polyfit(x, y, 1)[0] * 12  # ppm/mes -> ppm/año
        slopes.append(slope)
    return pd.Series(slopes, index=s.index[window//2:-window//2])

slopes = rolling_slope(trend, window=120)

pre = slopes.loc[:'1999']
post = slopes.loc['2000':]

ic_pre = ic_media_bootstrap(pre.values)
ic_post = ic_media_bootstrap(post.values)
ttest = t_test_welch(post.values, pre.values)

print(f"Pendiente pre-2000:  {pre.mean():.3f} ppm/año  IC95%: [{ic_pre[0]:.3f}, {ic_pre[1]:.3f}]")
print(f"Pendiente post-2000: {post.mean():.3f} ppm/año  IC95%: [{ic_post[0]:.3f}, {ic_post[1]:.3f}]")
print(f"Δ pendiente: {ttest['diff']:.3f} ppm/año²  IC95%: [{ttest['ci'][0]:.3f}, {ttest['ci'][1]:.3f}]  p={ttest['p']:.2e}")

> **Lectura guiada por la evidencia (dato → método → hallazgo):** La serie observada de Mauna Loa combina un ciclo anual muy marcado (biosfera) con una tendencia de largo plazo creciente. Por eso el análisis no se apoya en la curva cruda sino en (i) descomposición STL para separar tendencia, estacionalidad y residuo, (ii) tasas de crecimiento interanual y pendientes móviles para cuantificar cambios en el régimen, y (iii) detección de quiebres/ajuste por tramos y validación temporal para distinguir aceleración persistente de fluctuaciones. Las conclusiones que siguen se formulan a partir de las métricas calculadas en las celdas anteriores y de sus salidas gráficas/tabulares.

**Qué debe observarse en las salidas previas:**
1. **Escala y continuidad:** rango temporal, nivel inicial/final y completitud de la serie (`series`, `stl`).
2. **Estacionalidad vs tendencia:** magnitud del componente estacional y proporción de variabilidad explicada frente al residuo.
3. **Cambio de régimen:** evolución de la pendiente/tasa suavizada y fecha(s) de quiebre sugeridas por los métodos aplicados (o fallback documentado si algún método no está disponible).
4. **Adecuación y predictibilidad:** ajuste por tramos, métricas de validación temporal (MAE/MASE cuando se calculan) y diagnósticos de residuos para acotar la confianza inferencial.

In [ ]:
# Registrar hallazgos con narrative helpers (dinámicos, alineados a resultados ejecutados)
from narrative import hallazgo
from IPython.display import display, Markdown
import numpy as np
import pandas as pd

# Estacionalidad: amplitud y proporción de varianza explicada
seasonal = stl['seasonal'].dropna()
trend_c = stl['trend'].dropna()
resid_c = stl['resid'].dropna()
obs_c = stl['observed'].dropna()
amp_ppm = float(seasonal.max() - seasonal.min())
var_obs = float(np.nanvar(obs_c, ddof=1))
var_seasonal = float(np.nanvar(seasonal, ddof=1))
seasonal_share = (var_seasonal / var_obs) if var_obs > 0 else np.nan

if np.isfinite(seasonal_share):
    display(hallazgo(
        f"Estacionalidad anual marcada en la serie analizada: amplitud pico-a-valle ≈ {amp_ppm:.2f} ppm; la varianza del componente estacional representa ~{seasonal_share*100:.1f}% de la varianza de la serie observada (descomposición STL).",
        evidencia="STL (period=12): var(seasonal)/var(observed) y rango del componente estacional",
        ic=None
    ))
else:
    display(Markdown("**Hallazgo:** No fue posible cuantificar de forma estable la participación de la estacionalidad en la varianza."))

# Cambio en tasa de crecimiento (pendientes móviles pre/post 2000, según ventanas disponibles)
pre_mean = float(np.nanmean(pre.values)) if len(pre) else np.nan
post_mean = float(np.nanmean(post.values)) if len(post) else np.nan
if np.isfinite(pre_mean) and np.isfinite(post_mean):
    txt = (
        f"Las pendientes móviles estimadas sobre la tendencia muestran un cambio entre el periodo pre-2000 y post-2000: "
        f"media pre-2000 = {pre_mean:.3f} ppm/año (IC95% bootstrap [{ic_pre[0]:.3f}, {ic_pre[1]:.3f}]) frente a "
        f"media post-2000 = {post_mean:.3f} ppm/año (IC95% bootstrap [{ic_post[0]:.3f}, {ic_post[1]:.3f}]). "
        f"Diferencia (post-pre) = {ttest['diff']:.3f} ppm/año (IC95% [{ttest['ci'][0]:.3f}, {ttest['ci'][1]:.3f}], p={ttest['p']:.3g})."
    )
    display(hallazgo(
        txt,
        evidencia="Pendiente OLS en ventanas móviles de 10 años sobre tendencia STL + bootstrap + t-test de Welch",
        ic=(float(ttest['ci'][0]), float(ttest['ci'][1])),
        pval=float(ttest['p']) if np.isfinite(ttest['p']) else None
    ))
else:
    display(Markdown("**Hallazgo:** No fue posible estimar de forma robusta la comparación de pendientes pre/post con las ventanas disponibles."))

# Breakpoint seleccionado (método y fecha efectivamente usados)
method_bp = 'PELT (ruptures)' if ('breaks_pelt' in globals() and breaks_pelt and len(breaks_pelt) >= 2) else ('Bai-Perron (statsmodels)' if ('bp_breaks' in globals() and bp_breaks) else 'Referencia/fallback')
bp_idx_val = int(bp_idx) if 'bp_idx' in globals() and bp_idx is not None else None
if 'bp_date' in globals() and pd.notna(pd.to_datetime(bp_date, errors='coerce')):
    bp_date_val = pd.to_datetime(bp_date)
elif bp_idx_val is not None and 'trend' in globals() and len(trend) > 0:
    bp_date_val = pd.to_datetime(trend.index[min(max(bp_idx_val, 0), len(trend)-1)])
else:
    bp_date_val = pd.NaT
if pd.notna(bp_date_val) and bp_idx_val is not None:
    bp_txt = f"Quiebre estructural utilizado en el ajuste por tramos: {bp_date_val.date()} (índice {bp_idx_val} en la tendencia), según método: {method_bp}."
else:
    bp_txt = f"No se identificó un quiebre estructural único robusto en esta ejecución (método reportado: {method_bp}); se documenta el resultado disponible sin forzar una fecha."
if method_bp == 'Referencia/fallback':
    bp_txt += " Interpretación cautelosa: al no disponer de un quiebre estimado por los métodos anteriores en este entorno/serie, se usa un punto de referencia para ilustrar el contraste por regímenes."
display(hallazgo(bp_txt, evidencia=f"Selección de breakpoint en tendencia STL ({method_bp})", ic=None))


## 4. Dos regímenes, una atribución pendiente (1 cuartilla)

In [ ]:
# OPCIÓN A: Piecewise linear regression con breakpoint estimado/seleccionado
from stats_helpers import piecewise_linear_regression

# Usar breakpoint seleccionado en la celda anterior
bp_date = trend.index[bp_idx]
print(f"Breakpoint usado: {bp_date.date()}")

x = np.arange(len(trend))
y = trend.values
model_pw = piecewise_linear_regression(x, y, [bp_idx])
print(model_pw.summary())

# Predicciones
pred_pw = model_pw.predict(model_pw.model.exog)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trend.index, trend.values, alpha=0.5, label='Tendencia STL')
ax.plot(trend.index, pred_pw, color='#d62728', linewidth=2, label='Piecewise linear (2 segmentos)')
ax.axvline(bp_date, color='gray', linestyle=':', linewidth=2, label=f'Breakpoint: {bp_date.year}')
ax.set_ylabel('CO₂ (ppm)')
ax.set_title('Modelo Piecewise Linear: dos regímenes de crecimiento')
ax.legend()
plt.tight_layout()
save_fig("01_piecewise_fit")
plt.show()


In [ ]:
# OPCIÓN B: Prophet con changepoint detection nativo
try:
    from prophet import Prophet
    
    df_prophet = trend.reset_index()
    df_prophet.columns = ['ds', 'y']
    
    m = Prophet(changepoint_prior_scale=0.05, yearly_seasonality=False,
                weekly_seasonality=False, daily_seasonality=False)
    m.fit(df_prophet)
    
    future = m.make_future_dataframe(periods=60, freq='M')
    forecast = m.predict(future)
    
    fig = m.plot(forecast)
    plt.title('Prophet: tendencia + changepoints + forecast 5 años')
    save_fig("01_prophet_forecast")
    plt.show()
    
    # Changepoints detectados por Prophet
    changepoints = m.changepoints
    print(f"Prophet changepoints: {len(changepoints)} detectados")
    for cp in changepoints:
        print(f"  {cp.date()}")
except ImportError:
    print("Prophet no instalado - saltando")

In [ ]:
# Validación: Rolling origin forecast (últimos 10 años)
from stats_helpers import rolling_origin_forecast, mase_score

def arima_forecaster(train):
    from statsmodels.tsa.arima.model import ARIMA
    model = ARIMA(train, order=(1,1,1))
    fitted = model.fit()
    return fitted.forecast(12)

cv_results = rolling_origin_forecast(trend, arima_forecaster, window=240, horizon=12, step=12)
if len(cv_results) > 0:
    mase = mase_score(cv_results['actual'], cv_results['forecast'], seasonal_period=12)
    mae = cv_results['abs_error'].mean()
    print(f"Rolling CV (ARIMA 1,1,1): MAE={mae:.3f} ppm, MASE={mase:.3f}")
    print(f"N forecasts: {len(cv_results)}")
else:
    print("CV falló - datos insuficientes")

In [ ]:
# Diagnósticos de residuos del modelo piecewise
from stats_helpers import ts_diagnostics

resid_pw = y - pred_pw
diag = ts_diagnostics(resid_pw, lags=24)

print("=== DIAGNÓSTICOS RESIDUOS PIECEWISE ===")
for k, v in diag.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Gráficos diagnósticos
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0,0].plot(trend.index, resid_pw, alpha=0.7)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Residuos en el tiempo')

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plot_acf(resid_pw, lags=36, ax=axes[0,1])
axes[0,1].set_title('ACF residuos')
plot_pacf(resid_pw, lags=36, ax=axes[1,0])
axes[1,0].set_title('PACF residuos')

from stats_helpers import qq_plot
qq_plot(resid_pw, ax=axes[1,1])

plt.tight_layout()
save_fig("01_residuals_diagnostics")
plt.show()

> **Resumen técnico (trazabilidad dato → método → hallazgo):** La evidencia proviene de la serie mensual de Mauna Loa efectivamente cargada (`data/raw/co2_mm_mlo.csv`) y de transformaciones explícitas: (1) descomposición STL para separar señal tendencial y estacional, (2) estimación de tasas/pendientes y contraste pre/post quiebre mediante ventanas móviles e inferencia bootstrap/Welch, (3) ajuste *piecewise* alrededor del quiebre seleccionado y (4) evaluación complementaria con validación temporal tipo *rolling-origin* (ARIMA base, cuando es factible) y diagnósticos de residuos.

La interpretación se limita a lo que muestran estas salidas: dirección y magnitud del cambio estimado, fecha de quiebre utilizada, ajuste y errores de pronóstico observados, y cumplimiento (o no) de supuestos en residuos. No se atribuye causalidad a partir de la serie univariada; el quiebre estadístico se trata como evidencia descriptiva/inferencial que requiere contrastación con fuentes externas (emisiones, forzantes) para explicación causal.

## 5. Lo que *no* podemos decir (¼ cuartilla)

In [ ]:
# Limitaciones explícitas (con evidencia de diagnósticos cuando está disponible)
from narrative import limitacion
from IPython.display import display
import numpy as np

display(limitacion(
    "Registro único (Mauna Loa, 3.4 km elevación): no equivale a promedio global; pueden existir diferencias sistemáticas respecto a series globales integradas.",
    "media",
    "Contrastar con promedios globales NOAA/otras redes y con productos armonizados para evaluar robustez de la magnitud estimada."
))

display(limitacion(
    "Tratamiento de datos faltantes/interpolación (y elección de la serie usada para modelar) puede atenuar extremos o afectar estimaciones locales de variabilidad.",
    "baja",
    "Repetir análisis con mediciones observadas y con métodos explícitos para missingness (p. ej. modelos de estado/Kalman) como sensibilidad."
))

display(limitacion(
    "La detección de quiebres y la partición en regímenes dependen de supuestos, hiperparámetros y disponibilidad de métodos en el entorno; un quiebre estadístico no implica causalidad.",
    "alta",
    "Comparar especificaciones (ventanas/penalizaciones/modelos) y contrastar con información externa (emisiones, uso de suelo, forzantes) antes de atribuir mecanismos."
))

bp_p = diag.get('breusch_pagan_p', np.nan) if 'diag' in globals() and isinstance(diag, dict) else np.nan
if np.isfinite(bp_p):
    sev = "alta" if bp_p < 0.01 else ("media" if bp_p < 0.05 else "baja")
    txt_h = f"Diagnóstico de heterocedasticidad en residuos del ajuste por tramos: Breusch-Pagan p={bp_p:.4g}."
    if bp_p < 0.05:
        txt_h += " Sugiere varianza no constante; los intervalos/errores estándar pueden verse afectados."
    else:
        txt_h += " No se observa evidencia fuerte de heterocedasticidad a niveles usuales, aunque persiste la posibilidad de otros incumplimientos."
else:
    sev = "media"
    txt_h = "No fue posible evaluar completamente la heterocedasticidad de residuos en esta ejecución."
display(limitacion(txt_h, sev, "Usar errores robustos (HAC/Newey-West), modelado de varianza (p. ej. GARCH) o especificaciones alternativas si se detectan desviaciones."))


## 6. Cierre argumental y trazabilidad (dato → método → hallazgo → límite)

**Desarrollo argumental (sustantivo):** Esta historia parte de un dato observacional concreto: la serie mensual de CO₂ en Mauna Loa incluida en `data/raw/co2_mm_mlo.csv`. El dato impone la estrategia analítica: (1) su estacionalidad obliga a descomponer antes de interpretar tendencia; (2) su extensión temporal permite estimar tasas de cambio y comparar regímenes; (3) sus propiedades (posibles faltantes/interpolaciones, autocorrelación) obligan a validar supuestos y a leer con cautela la inferencia. El recorrido no es decorativo: cada método responde a una característica del dataset y cada salida (componentes STL, pendientes/IC, quiebre seleccionado, ajuste por tramos, validación y diagnósticos) delimita qué puede afirmarse y con qué incertidumbre.

**Trazabilidad explícita:** la pregunta (“¿cambia la tasa de crecimiento?”) se operacionaliza en estimadores calculados en el notebook; los hallazgos se reportan desde esos estimadores; las limitaciones se derivan tanto del origen del dato (estación única, procesamiento de la fuente) como del comportamiento de los residuos y de la sensibilidad metodológica. Así, la narrativa se sostiene en evidencia producida a partir del dataset analizado y no en cifras externas prefijadas.

### Nota de coherencia narrativa y extensión
La historia mantiene coherencia con el arquetipo de serie temporal/cambio estructural porque el argumento central (régimen y variación de la tasa) solo es defendible mediante descomposición, cuantificación temporal y contraste de modelos sobre la serie concreta utilizada. Si se eliminara esa cadena dato→método→resultado, el relato perdería su base empírica.

In [ ]:
# Conclusión estructurada (dinámica, basada en resultados calculados)
from narrative import conclusion_tecnica
from IPython.display import display
import numpy as np
import pandas as pd

pre_mean = float(np.nanmean(pre.values)) if len(pre) else np.nan
post_mean = float(np.nanmean(post.values)) if len(post) else np.nan
delta = float(ttest['diff']) if 'ttest' in globals() else np.nan
ci0 = float(ttest['ci'][0]) if 'ttest' in globals() else np.nan
ci1 = float(ttest['ci'][1]) if 'ttest' in globals() else np.nan
pval = float(ttest['p']) if 'ttest' in globals() else np.nan

# R2 del ajuste por tramos
r2_pw = np.nan
try:
    r2_pw = float(model_pw.rsquared)
except Exception:
    pass

bp_txt = pd.to_datetime(bp_date).date() if 'bp_date' in globals() else 'N/D'
method_bp = 'PELT' if (('breaks_pelt' in globals()) and breaks_pelt and len(breaks_pelt) >= 2) else ('Bai-Perron' if (('bp_breaks' in globals()) and bp_breaks) else 'referencia/fallback')

cv_txt = "no disponible en esta ejecución"
try:
    if 'cv_results' in globals() and len(cv_results) > 0:
        mae_v = float(cv_results['abs_error'].mean())
        mase_v = float(mase_score(cv_results['actual'], cv_results['forecast'], seasonal_period=12)) if 'mase_score' in globals() else np.nan
        if np.isfinite(mase_v):
            cv_txt = f"MAE={mae_v:.3f} ppm, MASE={mase_v:.3f} (ARIMA(1,1,1), rolling-origin)"
        else:
            cv_txt = f"MAE={mae_v:.3f} ppm (ARIMA(1,1,1), rolling-origin)"
except Exception:
    pass

respuesta = (
    f"Sobre la serie de CO₂ analizada (tendencia STL), las pendientes móviles indican un cambio de {pre_mean:.3f} ppm/año (pre-2000) a {post_mean:.3f} ppm/año (post-2000), "
    f"con Δ={delta:.3f} ppm/año (IC95% [{ci0:.3f}, {ci1:.3f}], p={pval:.3g}). "
    f"El ajuste por tramos se realizó con quiebre en {bp_txt} (método: {method_bp}); R² del modelo por tramos sobre la tendencia: {r2_pw:.4f} cuando está disponible. "
    f"Validación temporal: {cv_txt}. La evidencia respalda un cambio en la tasa dentro del periodo observado, sujeto a las limitaciones y supuestos diagnosticados."
)

display(conclusion_tecnica(
    respuesta=respuesta,
    siguiente_paso="Contrastar el quiebre/tasas con series externas (promedio global NOAA, inventarios de emisiones y forzantes) y evaluar sensibilidad a preprocesamiento, ventana de estimación y especificación del modelo antes de cualquier atribución causal.",
    reproducibilidad="Reproducible en el entorno núcleo declarado: `python3 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt && python build_all.py` (o ejecución secuencial de notebooks con `data/raw/` disponible). Resultados avanzados opcionales (p. ej. Prophet/ruptures) dependen de dependencias no incluidas en el núcleo."
))
